# 1. Lakehouse and semantic layer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammedelsayed741996-blip/berlin-data-portfolio/blob/main/b2b-saas-data-platforms/agentic-lakehouse-reference-architecture/notebooks/01_lakehouse_and_semantic_layer.ipynb)

Kestrel Industrial Supply is a fictional distributor; every account, invoice and person here is synthetic. This notebook rebuilds the bronze, silver and gold layers from two simulated ERP exports, runs the reconciliation checks, and shows how the semantic layer turns a metric request into SQL with row-level security.

In [1]:
import os, sys, subprocess, warnings
from pathlib import Path

REPO = "https://github.com/mohammedelsayed741996-blip/berlin-data-portfolio.git"
PROJECT = "b2b-saas-data-platforms/agentic-lakehouse-reference-architecture"

if "google.colab" in sys.modules:
    if not Path("/content/berlin-data-portfolio").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/berlin-data-portfolio"], check=True)
    os.chdir(f"/content/berlin-data-portfolio/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jsonschema"], check=True)
else:
    here = Path.cwd()
    while not (here / "src" / "agentlake").exists() and here != here.parent:
        here = here.parent
    os.chdir(here)

sys.path.insert(0, str(Path.cwd() / "src"))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 80)
from IPython.display import Image, display
print("Project folder:", Path.cwd().name)

Project folder: agentic-lakehouse-reference-architecture


In [2]:
from agentlake import lakehouse
out = lakehouse.build_all(Path("lake"))
print("Problems planted in bronze:", out["planted"])
out["quality"]

Problems planted in bronze: {'duplicate_invoices': 25, 'missing_due_dates': 72}


,check,passed,detail
0,silver invoice ids are unique,True,2757 rows
1,silver keeps every source invoice,True,2757 vs 2757
2,silver amounts equal source in EUR (to the cent),True,
3,no missing due dates in silver,True,
4,gold open receivables equal source,True,"4,286,062.17 vs 4,286,062.17"
5,every account has an owner,True,


Every check compares a gold number with the source system, so a silver rule that drops or double-counts invoices fails the build. The gold layer is what the agent may read:

In [3]:
gold = {f.stem: pd.read_csv(f) for f in sorted(Path("lake/gold").glob("*.csv"))}
for name, df in gold.items():
    print(f"{name:16s} {len(df):5d} rows   columns: {', '.join(df.columns[:8])}{' ...' if len(df.columns) > 8 else ''}")
gold["account_ar"].sort_values("overdue_eur", ascending=False).head(5)[["account_id", "name", "region", "owner", "open_eur", "overdue_eur", "credit_used_pct"]]

account_ar          60 rows   columns: account_id, name, region, segment, owner, payment_terms_days, credit_limit_eur, open_eur ...
open_invoices      388 rows   columns: invoice_id, account_id, owner, region, issued, due, amount_eur, open_eur ...
revenue_monthly   2805 rows   columns: month, account_id, owner, region, category, revenue_eur


,account_id,name,region,owner,open_eur,overdue_eur,credit_used_pct
17,A018,Elbtal Anlagentechnik AG,DACH,AM06,182712.33,96296.13,0.65
24,A025,Havelstahl Industrieservice KG,DACH,AM01,230940.63,93677.85,0.81
53,A054,Al Noor Trading LLC,Gulf,AM06,195768.90,69612.93,0.70
0,A001,Alpenrand Maschinenbau GmbH,DACH,AM01,152559.96,49649.40,1.88
12,A013,Oderblick Fertigung GmbH,DACH,AM01,60225.84,48082.05,0.29


## The semantic layer

The agent asks for a metric, dimensions and filters from allow-lists. The layer writes the SQL, binds every value as a parameter and adds the owner predicate for account managers.

In [4]:
from agentlake import semantic as S
sql, params = S.compile_metric("overdue_receivables", ["region"], [{"column": "segment", "op": "eq", "value": "OEM"}],
                               S.Scope("AM02", "AM02"))
print(sql); print("params:", params)

SELECT region, ROUND(SUM(overdue_eur), 2) AS value FROM account_ar WHERE owner = ? AND segment = ? GROUP BY region ORDER BY value DESC LIMIT 20
params: ['AM02', 'OEM']


In [5]:
wh = S.Warehouse(Path("lake/gold"))
am = S.Scope("AM02", "AM02"); fin = S.Scope("FIN01", None)
print("Account manager AM02 sees:", wh.run(*S.compile_metric("open_receivables", None, None, am)).iloc[0, 0].round(2))
print("Finance sees:            ", wh.run(*S.compile_metric("open_receivables", None, None, fin)).iloc[0, 0].round(2))

Account manager AM02 sees: 380144.76
Finance sees:             4286062.17


Anything outside the allow-lists is refused before any SQL exists, including a filter value that tries to smuggle SQL in:

In [6]:
for metric, dims, filters in [("salary", None, None), ("revenue", ["contact_email"], None),
                              ("open_receivables", None, [{"column": "region", "op": "eq", "value": "DACH' OR '1'='1"}])]:
    try:
        sql, params = S.compile_metric(metric, dims, filters, am)
        print("compiled, value bound as a parameter:", params, "->", len(wh.run(sql, params)), "row(s)")
    except S.SemanticError as e:
        print("refused:", e)

refused: unknown metric 'salary'; allowed: open_receivables, overdue_receivables, accounts, revenue, open_invoice_count, open_invoice_amount
refused: 'contact_email' is not a dimension of revenue; allowed: month, account_id, owner, region, category
compiled, value bound as a parameter: ['AM02', "DACH' OR '1'='1"] -> 1 row(s)
